# Notebook 03 — Feature Engineering & Category Mapping

**Project:** Starbucks Customer Voice Intelligence: A U.S. Coffee Chain Market Study  
**Purpose:** Add derived columns to the cleaned dataset: brand segments, satisfaction tiers, time dimensions, reviewer activity tiers, topic tags, and VADER sentiment scores.

| Column | Source | Description |
|--------|--------|-------------|
| `star_tier` | `review_stars` | Critical / Neutral / Positive |
| `brand_category` | `name` | Starbucks / Dunkin' / Independent Café / ... |
| `review_length` | `text` | Word count |
| `length_tier` | `review_length` | Short / Medium / Long |
| `year`, `month`, `quarter`, `day_of_week` | `date` | Time decomposition |
| `year_quarter` | `date` | e.g. "2019Q3" |
| `is_weekend` | `date` | Saturday or Sunday |
| `user_activity_tier` | `user_review_count` | Casual / Regular / Power / Elite |
| `topic_tag` | `text` | Service / Food Quality / Price / Ambiance / Wait Time / Cleanliness / General |
| `sentiment_score` | `text` (VADER) | Compound score in [-1.0, 1.0] |
| `sentiment_label` | `sentiment_score` | Positive / Neutral / Negative |

## 0. Environment setup

In [1]:
import sys
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path(".").resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))

INTERIM_DIR = PROJECT_ROOT / "data" / "processed"

from src.mappings import apply_all_mappings
from src.nlp_utils import apply_sentiment

print("Environment ready.")

Environment ready.


## 1. Load cleaned dataset

Load `cleaned_joined.parquet` from Notebook 02.

In [2]:
df = pd.read_parquet(INTERIM_DIR / "cleaned_joined.parquet")

print(f"Loaded: {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"Date range: {df['date'].min().date()} -> {df['date'].max().date()}")

Loaded: 381,999 rows x 19 columns
Date range: 2017-01-01 -> 2021-12-31


## 2. Category mappings

Apply the category mapping functions from `src/mappings.py`. The raw columns cover what happened; these mappings add the dimensions needed to ask why and for whom.

In [3]:
df = apply_all_mappings(df)

print(f"\nDataset shape after mappings: {df.shape[0]:,} rows x {df.shape[1]} columns")

Applying category mappings...


  star_tier        → {'Positive': 274371, 'Critical': 72876, 'Neutral': 34752}
  brand_category   → {'Independent Café': 362334, 'Starbucks': 11675, "Dunkin'": 6866, 'Dutch Bros': 925, 'The Coffee Bean': 128, "Peet's Coffee": 71}
  length_tier      → {'Medium': 179158, 'Short': 144057, 'Long': 58784}
  user_tier        → {'Regular': 130773, 'Casual': 113510, 'Power': 87318, 'Elite': 50398}
  topic_tag        → {'Service': 229874, 'Food Quality': 92802, 'General': 40949, 'Ambiance': 8949, 'Price': 5881, 'Wait Time': 3121, 'Cleanliness': 423}

Dataset shape after mappings: 381,999 rows x 31 columns


In [4]:
# Spot-check: sample 3 rows per star_tier
print("=== star_tier spot-check ===")
for tier in ["Critical", "Neutral", "Positive"]:
    sample = df[df["star_tier"] == tier][["review_stars", "star_tier", "text"]].head(1)
    for _, row in sample.iterrows():
        print(f"  [{tier}] stars={row['review_stars']}  text={str(row['text'])[:80]}")
print()

# Spot-check: brand_category distribution
print("=== brand_category distribution ===")
print(df["brand_category"].value_counts().to_string())
print()

# Spot-check: topic_tag distribution
print("=== topic_tag distribution ===")
print(df["topic_tag"].value_counts().to_string())
print()

# Spot-check: user_activity_tier
print("=== user_activity_tier distribution ===")
print(df["user_activity_tier"].value_counts().to_string())
print()

# Spot-check: time fields
print("=== year distribution ===")
print(df["year"].value_counts().sort_index().to_string())

=== star_tier spot-check ===
  [Critical] stars=1.0  text=Pretty slow service and the waitresses aren't very kind. I guess I was expecting
  [Neutral] stars=3.0  text=Food is good, but the cafe itself is extremely dirty. I will probably not return
  [Positive] stars=5.0  text=First time there and it was excellent!!! It feels like your are entering someone

=== brand_category distribution ===
brand_category
Independent Café    362334
Starbucks            11675
Dunkin'               6866
Dutch Bros             925
The Coffee Bean        128
Peet's Coffee           71

=== topic_tag distribution ===
topic_tag
Service         229874
Food Quality     92802
General          40949
Ambiance          8949
Price             5881
Wait Time         3121
Cleanliness        423

=== user_activity_tier distribution ===
user_activity_tier
Regular    130773
Casual     113510
Power       87318
Elite       50398

=== year distribution ===
year
2017    83959
2018    92112
2019    90702
2020    53518
2021 

## 3. Sentiment scoring (VADER)

Score sentiment on all review texts using VADER, a rule-based model built for informal language. It returns a compound score in [-1.0, 1.0] and a Positive / Neutral / Negative label.

Star ratings and sentiment diverge more than you'd expect. A 3-star review can read as genuinely upset; a 5-star one can be almost indifferent. The sentiment score is a second read on tone, independent of what the customer clicked.

In [5]:
df = apply_sentiment(df, text_col="text")

print(f"\nSentiment label distribution:")
print(df["sentiment_label"].value_counts().to_string())

Running VADER sentiment scoring on 381,999 reviews...


  Sentiment distribution: {'Positive': 330902, 'Negative': 46733, 'Neutral': 4364}

Sentiment label distribution:
sentiment_label
Positive    330902
Negative     46733
Neutral       4364


## 4. Spot-check sentiment vs. star rating

Pull five reviews per sentiment class and check them against star ratings and the raw text. Better to catch a calibration issue here than in the analysis notebooks.

In [6]:
for label in ["Positive", "Neutral", "Negative"]:
    print(f"\n{'='*60}")
    print(f"  Sentiment: {label}")
    print(f"{'='*60}")
    sample = df[df["sentiment_label"] == label][["review_stars", "sentiment_score", "text"]].sample(5, random_state=42)
    for _, row in sample.iterrows():
        print(f"  stars={int(row['review_stars'])}  score={row['sentiment_score']:.3f}")
        print(f"  {str(row['text'])[:120]}")
        print()


  Sentiment: Positive
  stars=2  score=0.999
  UPDATE (2/18/18): We came back to give it another try. Same results: great food, once it arrived. It took more than an h

  stars=3  score=0.812
  It was ok. I'm originally from south Florida and It's probably the closest you can get to Cuba. I typed in Cuban bakery 

  stars=5  score=0.942
  Awesome experience!

They have wise assortment of baked goods including muffins, turnovers, cookies and cream cheese pie

  stars=4  score=0.930
  One of my favorite places to get pho,  it is always really fresh . I also love the crispy chicken, excellent!

  stars=3  score=0.996
  I have mixed feelings about this place. While I am very happy that Sweet Katie Bee's offers plenty of gluten free and ke


  Sentiment: Neutral
  stars=1  score=0.000
  I used to eat here on a regular basis. However, in the last several years the quality of the food has gone down consider

  stars=1  score=-0.026
  If i could leave no stars I would. Its 11:11pm, i need cof

In [7]:
# Star-sentiment alignment: avg sentiment score by star rating
print("Average sentiment score by star rating:")
print(df.groupby("review_stars")["sentiment_score"].mean().round(3).to_string())

Average sentiment score by star rating:
review_stars
1.0   -0.198
2.0    0.197
3.0    0.574
4.0    0.852
5.0    0.899


## 5. Save enriched dataset

Save as `analysis_ready.parquet`. Notebooks 05 through 14 all read from this file.

In [8]:
output_path = INTERIM_DIR / "analysis_ready.parquet"
df.to_parquet(output_path, index=False)

size_mb = output_path.stat().st_size / 1_048_576
print(f"Saved: {output_path.name}  ({size_mb:.1f} MB)")
print(f"Shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"\nFinal column list:")
for col in df.columns:
    print(f"  {col:<25} {str(df[col].dtype):<15} example: {str(df[col].iloc[0])[:40]}")

Saved: analysis_ready.parquet  (141.6 MB)
Shape: 381,999 rows x 33 columns

Final column list:
  review_id                 object          example: -P5E9BYUaK7s3PwBF5oAyg
  user_id                   object          example: Jha0USGDMefGFRLik_xFQg
  business_id               object          example: bMratNjTG5ZFEA6hVyr-xQ
  review_stars              float64         example: 5.0
  date                      datetime64[ns]  example: 2017-02-19 00:00:00
  text                      object          example: First time there and it was excellent!!!
  useful                    int64           example: 0
  funny                     int64           example: 0
  cool                      int64           example: 0
  name                      object          example: Portobello Cafe
  city                      object          example: Eddystone
  city_normalized           object          example: Eddystone
  state                     object          example: PA
  business_avg_stars        float64  

---

**Next: Notebook 04 — Pipeline Summary & Dataset Validation**